# Formula 1 — All Seasons Data Pipeline (OpenF1 → Google Sheets)

This notebook generalizes the single-season `2024_Formula_1_Season_Data.ipynb` /
`2025_Formula_1_Season_Data.ipynb` notebooks into one pipeline that:

1. Fetches Formula 1 data for **every season available** from the
   [OpenF1 API](https://openf1.org/) (not just one hardcoded year).
2. Cleans/shapes it into tidy tables.
3. Runs the same kind of exploratory SQL joins as the originals, but across all years.
4. Writes the data into a **Google Sheet** — this notebook targets
   [this spreadsheet](https://docs.google.com/spreadsheets/d/1FHF1yWmg5sv0e2WGVafTiM-zfzfaxvleRxG28nvjd9U) —
   with one tab per table: `Meetings`, `Sessions`, `Drivers`, `Session Results`,
   `Starting Grid`, `Laps`, `Pit Stops`.

**Prerequisites:**

- **The target spreadsheet must be shared (Editor access) with whichever Google
  account you authenticate as below.** It's not created for you — `SPREADSHEET_ID`
  in Step 2 points at the sheet linked above.
- **Running in Google Colab (recommended):** nothing else to set up. Step 3 below
  calls `google.colab.auth.authenticate_user()`, which pops up Colab's built-in
  Google sign-in — no API key, token, or secret to create or paste anywhere.
- **Running outside Colab** (local Jupyter, CI, etc.): either set
  `GOOGLE_APPLICATION_CREDENTIALS` to a service account JSON file that has edit
  access shared to it, or leave it unset and Step 3 will fall back to
  `gspread.oauth()`, which opens a browser-based Google sign-in flow once and
  caches the resulting token locally.
- Nothing else to install by hand — Step 1 below installs any missing packages
  automatically.

> Run the cells top-to-bottom. Steps 5–7 talk to the OpenF1 API (no auth needed).
> Step 10 talks to Google Sheets and needs the auth set up in Step 3.


## 1. Pre-Requisites

In [ ]:
# Auto-install any missing dependencies (safe to re-run; skips what's already present).
import importlib.util
import subprocess
import sys

_required_packages = ["pandas", "pandasql", "requests", "gspread"]
_missing = [pkg for pkg in _required_packages if importlib.util.find_spec(pkg) is None]
if _missing:
    print(f"Installing missing packages: {', '.join(_missing)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)

import json
import math
import os
import time
from datetime import datetime, timezone

import gspread
import pandas as pd
import requests
from pandasql import sqldf


## 2. Configuration

`YEARS_TO_INCLUDE` controls the scope of the pull:

- `None` → keep **every** season the OpenF1 API returns (fully "all years").
- A list like `[2023, 2024, 2025]` → restrict to those seasons only.

Laps (and pit stops) are the highest-volume tables — a full multi-year pull can be
tens/hundreds of thousands of rows. `PUSH_LAPS_TO_SHEETS` / `PUSH_PITS_TO_SHEETS` let
you keep them in the notebook's dataframes (e.g. to export as CSV/Parquet) without
writing that volume into the spreadsheet, if that's a concern.


In [ ]:
# --- Scope ---
YEARS_TO_INCLUDE = None          # e.g. [2023, 2024, 2025] to restrict; None = all seasons available

# --- OpenF1 API ---
OPENF1_BASE_URL = "https://api.openf1.org/v1"

# --- Google Sheets ---
# Target spreadsheet: https://docs.google.com/spreadsheets/d/1FHF1yWmg5sv0e2WGVafTiM-zfzfaxvleRxG28nvjd9U
SPREADSHEET_ID = "1FHF1yWmg5sv0e2WGVafTiM-zfzfaxvleRxG28nvjd9U"
SHEET_TABS = {
    "meetings": "Meetings",
    "sessions": "Sessions",
    "drivers": "Drivers",
    "session_results": "Session Results",
    "starting_grid": "Starting Grid",
    "laps": "Laps",
    "pit_stops": "Pit Stops",
}

PUSH_TO_SHEETS = True
PUSH_LAPS_TO_SHEETS = True
PUSH_PITS_TO_SHEETS = True
SHEETS_BATCH_SIZE = 5000   # rows per append_rows() call


## 3. Authenticate & Open the Spreadsheet

Resolution order for credentials:

1. **Colab** — `google.colab.auth.authenticate_user()` (interactive sign-in popup,
   nothing stored in this notebook or git history).
2. **Service account** — if `GOOGLE_APPLICATION_CREDENTIALS` points to a service
   account JSON file (non-Colab environments).
3. **Interactive OAuth** — `gspread.oauth()` as a last resort, opening a browser
   sign-in flow once and caching the token locally.

Then opens `SPREADSHEET_ID` (set in Step 2). This must be an existing spreadsheet the
authenticated account has **Editor** access to — sign in with the account it's shared
with, or share it with whichever account you're signing in as here.


In [ ]:
if not PUSH_TO_SHEETS:
    gc = None
    spreadsheet = None
else:
    try:
        from google.colab import auth as colab_auth
        colab_auth.authenticate_user()
        from google.auth import default as google_auth_default
        creds, _ = google_auth_default()
        gc = gspread.authorize(creds)
        print("Authenticated via Colab (google.colab.auth).")
    except ImportError:
        service_account_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
        if service_account_path:
            gc = gspread.service_account(filename=service_account_path)
            print(f"Authenticated via service account: {service_account_path}")
        else:
            gc = gspread.oauth()
            print("Authenticated via interactive OAuth (browser sign-in flow).")

    try:
        spreadsheet = gc.open_by_key(SPREADSHEET_ID)
        print(f"Opened spreadsheet: {spreadsheet.url}")
    except gspread.exceptions.APIError as e:
        raise RuntimeError(
            f"Could not open spreadsheet {SPREADSHEET_ID}. Make sure it exists and is "
            "shared (Editor access) with the account you authenticated as above."
        ) from e


## 4. Helper Functions

In [ ]:
def fetch_data_to_dataframe(url: str, df_name: str = "data", columns_to_omit: list = None,
                             verbose: bool = True) -> pd.DataFrame:
    """Fetch a JSON endpoint into a DataFrame. Returns an empty DataFrame on any failure."""
    if verbose:
        print(f"Attempting to fetch {df_name} from: {url}")
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        fetched_data = response.json()

        if fetched_data:
            df = pd.DataFrame(fetched_data)
            if verbose:
                print(f"Successfully fetched {len(df)} rows for {df_name}.")
            if columns_to_omit:
                existing_columns_to_drop = [c for c in columns_to_omit if c in df.columns]
                if existing_columns_to_drop:
                    df = df.drop(columns=existing_columns_to_drop)
            return df
        else:
            if verbose:
                print(f"No data found for {df_name} from the provided URL.")
            return pd.DataFrame()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {df_name} from {url}: {e}")
        return pd.DataFrame()


In [ ]:
def format_lap_duration(seconds):
    """Convert a lap duration in seconds (float) to an M:SS.mmm string."""
    if seconds is None or (isinstance(seconds, float) and math.isnan(seconds)):
        return None

    minutes, sec = divmod(seconds, 60)
    sec_int = int(sec)
    milliseconds = int(round((sec - sec_int) * 1000))

    return f"{int(minutes)}:{sec_int:02d}.{milliseconds:03d}"


# Run ad-hoc SQL against the dataframes in this notebook's global scope
pysqldf = lambda q: sqldf(q, globals())


def dataframe_to_sheet_values(df: pd.DataFrame) -> list:
    """Convert a DataFrame to a list-of-lists Sheets accepts: NaN -> '', dates -> ISO
    strings, numpy scalars -> native Python types."""
    rows = json.loads(df.to_json(orient="values", date_format="iso"))
    return [["" if v is None else v for v in row] for row in rows]


def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


In [ ]:
def get_or_create_worksheet(title: str, num_data_rows: int, num_cols: int):
    try:
        return spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        return spreadsheet.add_worksheet(title=title, rows=max(num_data_rows + 10, 100),
                                          cols=max(num_cols, 1))


def push_dataframe_to_sheet(df: pd.DataFrame, sheet_key: str, max_retries: int = 5) -> int:
    """Full-refresh the mapped worksheet from a DataFrame: clears it, writes the header
    row, then appends all rows in batches.

    Google Sheets has no native per-row upsert API (unlike Airtable), so this always
    leaves the sheet matching the dataframe exactly -- idempotent across re-runs, but
    any manual edits made directly in the sheet get overwritten.
    """
    title = SHEET_TABS[sheet_key]
    if df.empty:
        print(f"Skipping {title}: no rows to push.")
        return 0

    headers = df.columns.tolist()
    worksheet = get_or_create_worksheet(title, len(df), len(headers))
    values = dataframe_to_sheet_values(df)

    def _with_retry(fn, *args, **kwargs):
        attempt = 0
        while True:
            try:
                return fn(*args, **kwargs)
            except gspread.exceptions.APIError as e:
                attempt += 1
                if attempt > max_retries:
                    raise
                print(f"  Sheets API error on '{title}' (attempt {attempt}/{max_retries}): {e}. Retrying...")
                time.sleep(2 ** attempt)

    _with_retry(worksheet.clear)
    _with_retry(worksheet.update, [headers], value_input_option="RAW")

    pushed = 0
    for batch in chunked(values, SHEETS_BATCH_SIZE):
        _with_retry(worksheet.append_rows, batch, value_input_option="RAW")
        pushed += len(batch)

    print(f"Pushed {pushed} rows into '{title}'.")
    return pushed


## 5. Fetch Core Reference Data (all seasons, one call each)

The OpenF1 endpoints below return their **entire history** when called without a
year/date filter — not just the current season — so each of these only needs to be
fetched once, regardless of how many seasons we ultimately keep.


In [ ]:
sessions_url        = f"{OPENF1_BASE_URL}/sessions"
meetings_url         = f"{OPENF1_BASE_URL}/meetings"
drivers_url          = f"{OPENF1_BASE_URL}/drivers"
session_results_url  = f"{OPENF1_BASE_URL}/session_result"
starting_grid_url    = f"{OPENF1_BASE_URL}/starting_grid"
pits_url             = f"{OPENF1_BASE_URL}/pit"

sessions_df         = fetch_data_to_dataframe(sessions_url, df_name="sessions data (all years)")
meetings_df         = fetch_data_to_dataframe(meetings_url, df_name="meetings data (all years)")
drivers_df          = fetch_data_to_dataframe(drivers_url, df_name="drivers data (all years)")
session_results_df  = fetch_data_to_dataframe(session_results_url, df_name="session result data (all years)", verbose=False)
starting_grid_df    = fetch_data_to_dataframe(starting_grid_url, df_name="starting grid data (all years)", verbose=False)
pits_df             = fetch_data_to_dataframe(pits_url, df_name="pit stop data (all years)", verbose=False)

print("\nAvailable seasons in sessions data:", sorted(sessions_df["year"].dropna().unique().tolist()))


## 6. Scope to Selected Years (optional filter)

Everything downstream is derived from `sessions_df`'s `year` column, so this is the
single place that enforces `YEARS_TO_INCLUDE`.


In [ ]:
if YEARS_TO_INCLUDE:
    sessions_scoped = sessions_df[sessions_df["year"].isin(YEARS_TO_INCLUDE)].copy()
else:
    sessions_scoped = sessions_df.copy()

race_sessions_all = sessions_scoped[sessions_scoped["session_type"] == "Race"].copy()
race_sessions      = race_sessions_all  # kept for parity with the single-season notebooks' naming

scoped_meeting_keys  = set(sessions_scoped["meeting_key"].dropna().unique())
scoped_session_keys  = set(sessions_scoped["session_key"].dropna().unique())

meetings_scoped        = meetings_df[meetings_df["meeting_key"].isin(scoped_meeting_keys)].copy()
drivers_scoped         = drivers_df[drivers_df["session_key"].isin(scoped_session_keys)].copy()
session_results_scoped = session_results_df[session_results_df["session_key"].isin(scoped_session_keys)].copy()
starting_grid_scoped   = starting_grid_df[starting_grid_df["session_key"].isin(scoped_session_keys)].copy()
pits_scoped            = pits_df[pits_df["session_key"].isin(scoped_session_keys)].copy() if not pits_df.empty else pits_df

print(f"Sessions in scope: {len(sessions_scoped)}")
print(f"Race sessions in scope: {len(race_sessions_all)}")
print(f"Meetings in scope: {len(meetings_scoped)}")
print(f"Drivers rows in scope: {len(drivers_scoped)}")
print(f"Session results in scope: {len(session_results_scoped)}")
print(f"Starting grid rows in scope: {len(starting_grid_scoped)}")
print(f"Pit stop rows in scope: {len(pits_scoped)}")


## 7. Fetch Laps Data Per Race Session (looped, all selected years)

Laps are too high-volume for a single unfiltered call, so — exactly like the original
notebooks — we loop `session_key` by `session_key` over every race (incl. sprint) session
currently in scope, across every season.


In [ ]:
all_laps = []
columns_to_exclude = ["segments_sector_1", "segments_sector_2", "segments_sector_3"]

for i, (idx, row) in enumerate(race_sessions_all.iterrows(), start=1):
    session_key = row["session_key"]
    laps_url = f"{OPENF1_BASE_URL}/laps?session_key={session_key}"

    laps = fetch_data_to_dataframe(laps_url, df_name=f"laps in session {session_key}",
                                    columns_to_omit=columns_to_exclude, verbose=False)

    if laps is not None and not laps.empty:
        laps["session_key"] = session_key
        laps["meeting_key"] = row["meeting_key"]
        laps["year"] = row["year"]
        all_laps.append(laps)

    if i % 25 == 0 or i == len(race_sessions_all):
        print(f"Fetched laps for {i}/{len(race_sessions_all)} race sessions...")

if all_laps:
    laps_df = pd.concat(all_laps, ignore_index=True)
    laps_df["lap_duration_formatted"] = laps_df["lap_duration"].apply(format_lap_duration)
    print(f"\nTotal laps collected across all seasons in scope: {len(laps_df)}")
else:
    laps_df = pd.DataFrame()
    print("No laps data found for the selected scope.")


## 8. Shape Data for Google Sheets

Build the composite/natural key each table uses to identify a row (kept as a plain
column since Sheets has no native primary key), rename columns to the headers each
tab will use, and normalize types (booleans, NaN → blank, dates → ISO strings).


In [ ]:
# --- Meetings ---
meetings_sheet = meetings_scoped.rename(columns={
    "meeting_key": "Meeting Key",
    "meeting_name": "Meeting Name",
    "meeting_official_name": "Meeting Official Name",
    "location": "Location",
    "country_name": "Country Name",
    "country_code": "Country Code",
    "circuit_short_name": "Circuit Short Name",
    "circuit_key": "Circuit Key",
    "date_start": "Date Start",
    "year": "Year",
})
meetings_sheet = meetings_sheet[[c for c in [
    "Meeting Key", "Meeting Name", "Meeting Official Name", "Location", "Country Name",
    "Country Code", "Circuit Short Name", "Circuit Key", "Date Start", "Year",
] if c in meetings_sheet.columns]]

# --- Sessions ---
sessions_sheet = sessions_scoped.rename(columns={
    "session_key": "Session Key",
    "meeting_key": "Meeting Key",
    "session_name": "Session Name",
    "session_type": "Session Type",
    "location": "Location",
    "country_name": "Country Name",
    "circuit_short_name": "Circuit Short Name",
    "date_start": "Date Start",
    "date_end": "Date End",
    "year": "Year",
})
sessions_sheet = sessions_sheet[[c for c in [
    "Session Key", "Meeting Key", "Session Name", "Session Type", "Location",
    "Country Name", "Circuit Short Name", "Date Start", "Date End", "Year",
] if c in sessions_sheet.columns]]

# --- Drivers (dedup one row per session+driver) ---
drivers_sheet = drivers_scoped.copy()
drivers_sheet["Driver Session Key"] = (
    drivers_sheet["session_key"].astype(str) + "_" + drivers_sheet["driver_number"].astype(str)
)
drivers_sheet = drivers_sheet.drop_duplicates(subset=["Driver Session Key"])
drivers_sheet = drivers_sheet.rename(columns={
    "driver_number": "Driver Number",
    "session_key": "Session Key",
    "meeting_key": "Meeting Key",
    "full_name": "Full Name",
    "name_acronym": "Name Acronym",
    "team_name": "Team Name",
    "team_colour": "Team Colour",
    "country_code": "Country Code",
})
sessions_year_lookup = sessions_scoped.set_index("session_key")["year"]
drivers_sheet["Year"] = drivers_sheet["Session Key"].map(sessions_year_lookup)
drivers_sheet = drivers_sheet[[c for c in [
    "Driver Session Key", "Driver Number", "Session Key", "Meeting Key", "Full Name",
    "Name Acronym", "Team Name", "Team Colour", "Country Code", "Year",
] if c in drivers_sheet.columns]]

# --- Session Results ---
results_sheet = session_results_scoped.copy()
results_sheet["Result Key"] = (
    results_sheet["session_key"].astype(str) + "_" + results_sheet["driver_number"].astype(str)
)
results_sheet = results_sheet.rename(columns={
    "session_key": "Session Key",
    "meeting_key": "Meeting Key",
    "driver_number": "Driver Number",
    "position": "Position",
    "number_of_laps": "Number Of Laps",
    "points": "Points",
    "dnf": "DNF",
    "dns": "DNS",
    "dsq": "DSQ",
    "duration": "Duration",
    "gap_to_leader": "Gap To Leader",
})
results_sheet["Gap To Leader"] = results_sheet["Gap To Leader"].astype(str)
results_sheet["Year"] = results_sheet["Session Key"].map(sessions_year_lookup)
results_sheet = results_sheet[[c for c in [
    "Result Key", "Session Key", "Meeting Key", "Driver Number", "Position",
    "Number Of Laps", "Points", "DNF", "DNS", "DSQ", "Duration", "Gap To Leader", "Year",
] if c in results_sheet.columns]]

# --- Starting Grid ---
grid_sheet = starting_grid_scoped.copy()
grid_sheet["Grid Key"] = (
    grid_sheet["session_key"].astype(str) + "_" + grid_sheet["driver_number"].astype(str)
)
grid_sheet = grid_sheet.rename(columns={
    "session_key": "Session Key",
    "meeting_key": "Meeting Key",
    "driver_number": "Driver Number",
    "position": "Position",
    "lap_duration": "Lap Duration",
})
grid_sheet["Year"] = grid_sheet["Session Key"].map(sessions_year_lookup)
grid_sheet = grid_sheet[[c for c in [
    "Grid Key", "Session Key", "Meeting Key", "Driver Number", "Position", "Lap Duration", "Year",
] if c in grid_sheet.columns]]

# --- Laps ---
if not laps_df.empty:
    laps_sheet = laps_df.copy()
    laps_sheet["Lap Key"] = (
        laps_sheet["session_key"].astype(str) + "_" +
        laps_sheet["driver_number"].astype(str) + "_" +
        laps_sheet["lap_number"].astype(str)
    )
    laps_sheet = laps_sheet.rename(columns={
        "session_key": "Session Key",
        "meeting_key": "Meeting Key",
        "driver_number": "Driver Number",
        "lap_number": "Lap Number",
        "date_start": "Date Start",
        "duration_sector_1": "Duration Sector 1",
        "duration_sector_2": "Duration Sector 2",
        "duration_sector_3": "Duration Sector 3",
        "i1_speed": "I1 Speed",
        "i2_speed": "I2 Speed",
        "st_speed": "St Speed",
        "is_pit_out_lap": "Is Pit Out Lap",
        "lap_duration": "Lap Duration",
        "lap_duration_formatted": "Lap Duration Formatted",
        "year": "Year",
    })
    laps_sheet = laps_sheet[[c for c in [
        "Lap Key", "Session Key", "Meeting Key", "Driver Number", "Lap Number", "Date Start",
        "Duration Sector 1", "Duration Sector 2", "Duration Sector 3", "I1 Speed", "I2 Speed",
        "St Speed", "Is Pit Out Lap", "Lap Duration", "Lap Duration Formatted", "Year",
    ] if c in laps_sheet.columns]]
else:
    laps_sheet = pd.DataFrame()

# --- Pit Stops ---
if not pits_scoped.empty:
    pits_sheet = pits_scoped.copy()
    pits_sheet["Pit Key"] = (
        pits_sheet["session_key"].astype(str) + "_" +
        pits_sheet["driver_number"].astype(str) + "_" +
        pits_sheet["lap_number"].astype(str)
    )
    pits_sheet = pits_sheet.rename(columns={
        "session_key": "Session Key",
        "meeting_key": "Meeting Key",
        "driver_number": "Driver Number",
        "lap_number": "Lap Number",
        "date": "Date",
        "pit_duration": "Pit Duration",
    })
    pits_sheet["Year"] = pits_sheet["Session Key"].map(sessions_year_lookup)
    pits_sheet = pits_sheet[[c for c in [
        "Pit Key", "Session Key", "Meeting Key", "Driver Number", "Lap Number", "Date",
        "Pit Duration", "Year",
    ] if c in pits_sheet.columns]]
else:
    pits_sheet = pd.DataFrame()

print("Shaped row counts:")
for name, df_ in [("Meetings", meetings_sheet), ("Sessions", sessions_sheet),
                   ("Drivers", drivers_sheet), ("Session Results", results_sheet),
                   ("Starting Grid", grid_sheet), ("Laps", laps_sheet),
                   ("Pit Stops", pits_sheet)]:
    print(f"  {name}: {len(df_)}")


## 9. Exploratory SQL Joins (across all years)

Same style of query as the original notebooks' "Joining Data Required for Dashboard"
section, generalized so `round` and standings are computed **per year** instead of being
hardcoded to a single season.


In [ ]:
driver_standings_query = """
WITH grand_prix_rounds AS (
    SELECT
        ra.meeting_key,
        ra.year,
        md.meeting_name,
        MIN(ra.date_start) AS grand_prix_date,
        ROW_NUMBER() OVER (PARTITION BY ra.year ORDER BY MIN(ra.date_start)) AS round
    FROM race_sessions_all AS ra
    LEFT JOIN meetings_df AS md
      ON md.meeting_key = ra.meeting_key
    WHERE ra.session_type = 'Race'
    GROUP BY ra.meeting_key, ra.year, md.meeting_name
)

SELECT
    gr.round,
    md.meeting_name AS grand_prix,
    sg.driver_number,
    dr.name_acronym,
    dr.last_name,
    dr.team_name,
    sg.position AS grid_position,
    rs.position AS final_position,
    rs.points,
    ra.year,
    ra.session_name
FROM race_sessions_all AS ra
LEFT JOIN grand_prix_rounds AS gr
  ON gr.meeting_key = ra.meeting_key AND gr.year = ra.year
LEFT JOIN meetings_df AS md
  ON md.meeting_key = ra.meeting_key
LEFT JOIN starting_grid_df AS sg
  ON sg.session_key = ra.session_key
LEFT JOIN session_results_df AS rs
  ON rs.session_key = ra.session_key AND rs.driver_number = sg.driver_number
LEFT JOIN drivers_df AS dr
  ON dr.driver_number = sg.driver_number AND dr.session_key = ra.session_key
ORDER BY ra.year, gr.round
"""

driver_standings = pysqldf(driver_standings_query)
driver_standings.head()


In [ ]:
laps_with_sessions_query = """
SELECT
    ra.year,
    md.meeting_name,
    md.meeting_key,
    ld.driver_number,
    dr.full_name,
    ld.lap_number,
    ld.is_pit_out_lap,
    ld.lap_duration,
    ld.lap_duration_formatted
FROM race_sessions_all AS ra
JOIN meetings_df AS md
  ON md.meeting_key = ra.meeting_key
JOIN laps_df AS ld
  ON ld.session_key = ra.session_key
JOIN drivers_df AS dr
  ON dr.driver_number = ld.driver_number AND dr.session_key = ld.session_key
"""

laps_with_sessions = pysqldf(laps_with_sessions_query) if not laps_df.empty else pd.DataFrame()
laps_with_sessions.head()


## 10. Push Data to Google Sheets

Full-refreshes each tab from its shaped table (clear → header → append in batches).
Safe to re-run: the sheet always ends up matching the dataframe exactly.


In [ ]:
if PUSH_TO_SHEETS:
    push_dataframe_to_sheet(meetings_sheet, "meetings")
    push_dataframe_to_sheet(sessions_sheet, "sessions")
    push_dataframe_to_sheet(drivers_sheet, "drivers")
    push_dataframe_to_sheet(results_sheet, "session_results")
    push_dataframe_to_sheet(grid_sheet, "starting_grid")

    if PUSH_LAPS_TO_SHEETS:
        push_dataframe_to_sheet(laps_sheet, "laps")
    else:
        print("Skipping Laps push (PUSH_LAPS_TO_SHEETS = False).")

    if PUSH_PITS_TO_SHEETS:
        push_dataframe_to_sheet(pits_sheet, "pit_stops")
    else:
        print("Skipping Pit Stops push (PUSH_PITS_TO_SHEETS = False).")

    # gspread creates a new spreadsheet with a blank default "Sheet1" tab; drop it once
    # our named tabs exist, but only if it's still empty (never touches real data).
    try:
        default_ws = spreadsheet.worksheet("Sheet1")
        if default_ws.title not in SHEET_TABS.values() and not default_ws.get_all_values():
            spreadsheet.del_worksheet(default_ws)
            print("Removed the blank default 'Sheet1' tab.")
    except gspread.WorksheetNotFound:
        pass

    print(f"\nSpreadsheet: {spreadsheet.url}")
else:
    print("PUSH_TO_SHEETS is False — nothing was written to Google Sheets.")


## 11. Summary

In [ ]:
print("Seasons in scope:", sorted(sessions_scoped["year"].dropna().unique().tolist()))
print(f"Meetings: {len(meetings_sheet)}")
print(f"Sessions: {len(sessions_sheet)}")
print(f"Driver-session rows: {len(drivers_sheet)}")
print(f"Session results: {len(results_sheet)}")
print(f"Starting grid rows: {len(grid_sheet)}")
print(f"Laps: {len(laps_sheet)}")
print(f"Pit stops: {len(pits_sheet)}")
if PUSH_TO_SHEETS and spreadsheet is not None:
    print(f"Spreadsheet: {spreadsheet.url}")
